# 1.0 Import libraries

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd

# 2.0 Load Data

In [7]:
compiled = pd.read_csv("/content/drive/MyDrive/lmf/data/2026_10_02_integration_of_breeding_agronomic_data_for_visualization_tool/gas_nutrition_categories_2026_10_01.csv")
yield_grasses =  pd.read_csv("/content/drive/MyDrive/lmf/data/2026_10_02_integration_of_breeding_agronomic_data_for_visualization_tool/yield_breeding_complete.csv")

In [8]:
compiled.head(2)

,id_lab,id,subset,no,requisitioner,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,...,ch4_ml_g_dm_incubated_24h,methane_intensity,ch4_ml_g_ndf_digested_24h,n_replicates_gas,ch4_category,tddm_category,lmf_category,lmf_category_rank,quartile,quartile_rank
0,Dieta-1-:-F263137-+-F261072-(40%/60%)-Estrella...,Dieta-1-:-F-263137-+-F-261072-(40%/60%)-Estrel...,5.0,NaN,ISABEL MOLINA/ EXP Camaras In vivo,NaN,NaN,94.6,8.97,91.03,...,17.29,48.82,1.91,6.0,NaN,NaN,NaN,NaN,NaN,NaN
1,Dieta-1_Exp1,Dieta-1-Exp-1,3.0,193.0,LMF-invivo,Dieta 1_Exp1,NaN,89.0,9.50,89.53,...,21.82,71.84,321.88,6.0,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
yield_grasses.head(2)

,trial,field,plot,rep,X,Y,lab_delivery_date,sampling_date,material,requisitioner,...,dry_matter_pct,height_cm,leaf_number,cover_m2,yield_g_m2,yield_kgDM_ha_cut,yield_ton_ha_cut,yield_ton_ha_year,net_seed_g_m2,seed_filling_pct
0,NaN,E1S,3.0,NaN,NaN,NaN,2026-03-30 00:00:00,2026-04-01 00:00:00,Interspecific U.,in vivo,...,0.319262,38.0,2.5,0.504060,346.585910,6875.889680,6.88,41.255338,NaN,NaN
1,NaN,E1S,1.0,NaN,NaN,NaN,2026-03-30 00:00:00,2026-03-25 00:00:00,Interspecific U.,in vivo,...,0.173403,60.0,4.0,0.721346,419.909447,5821.193256,5.82,34.927160,NaN,NaN


In [10]:
yield_grasses.columns

Index(['trial', 'field', 'plot', 'rep', 'X', 'Y', 'lab_delivery_date',
       'sampling_date', 'material', 'requisitioner', 'id', 'genotype', 'state',
       'comercial_name', 'type', 'biomass_g', 'dry_biomass_g',
       'dry_matter_pct', 'height_cm', 'leaf_number', 'cover_m2', 'yield_g_m2',
       'yield_kgDM_ha_cut', 'yield_ton_ha_cut', 'yield_ton_ha_year',
       'net_seed_g_m2', 'seed_filling_pct'],
      dtype='object')

In [11]:
#Columns processing
category_columns = ['trial', 'field', 'plot', 'rep', 'X', 'Y', 'lab_delivery_date',
       'sampling_date', 'material', 'requisitioner', 'id', 'genotype', 'state',
       'comercial_name', 'type']
numeric_columns = [ 'biomass_g', 'dry_biomass_g',
       'dry_matter_pct', 'height_cm', 'leaf_number', 'cover_m2', 'yield_g_m2',
       'yield_kgDM_ha_cut', 'yield_ton_ha_cut', 'yield_ton_ha_year']

for df in [yield_grasses]:
    for col in category_columns:
        if col in df.columns:
            df[col] = df[col].astype('category')
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

In [12]:
# Group by subset and id_lab, keeping categorical columns
def mean_with_replicates(df, category_columns, numeric_columns):

    # Categorical columns to retain (excluding grouping columns)
    cat_cols = [
        col for col in category_columns
        if col not in [ 'id']
    ]

    # Keep the first value of each categorical column within each group
    cat_df = (
        df.groupby([ 'id'])[cat_cols]
          .first()
    )

    # Calculate the mean of the numeric columns
    mean_df = (
        df.groupby(['id'])[numeric_columns]
          .mean()
          .round(2)
    )

    # Count the number of replicates in each group
    mean_df['n_replicates_agronomic'] = (
        df.groupby([ 'id'])
          .size()
    )

    # Combine categorical columns, means, and replicate counts
    result = (
        cat_df
        .join(mean_df)
        .reset_index()
    )

    return result

In [13]:
grasses_lab_yield = mean_with_replicates(yield_grasses, category_columns, numeric_columns)

/tmp/ipykernel_4531/2122244599.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id'])[cat_cols]
/tmp/ipykernel_4531/2122244599.py:18: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(['id'])[numeric_columns]
/tmp/ipykernel_4531/2122244599.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby([ 'id'])


# 3.0 Join

In [14]:
grasses_lab_yield = compiled.merge(grasses_lab_yield, on="id")

In [15]:
grasses_lab_yield

,id_lab,id,subset,no,requisitioner_x,tax_name,functional_group,dm_percentage,ash_dm,om_percentage,...,dry_biomass_g,dry_matter_pct,height_cm,leaf_number,cover_m2,yield_g_m2,yield_kgDM_ha_cut,yield_ton_ha_cut,yield_ton_ha_year,n_replicates_agronomic
0,F24-3582,CIAT-BR-02-1752,1.0,51.0,Breeding,Urochloa interespecific,Grasses,96.03,15.70,84.30,...,87.49,0.21,44.50,3.00,0.61,156.98,1667.99,1.67,10.00,9
1,F24-3583,CIAT-BR-02-1794,1.0,53.0,Breeding,Urochloa interespecific,Grasses,97.05,15.02,84.98,...,62.17,0.23,36.00,2.25,0.56,120.70,1207.01,1.21,7.23,6
2,F24-3584,CIAT-BR-06-0423,1.0,55.0,Breeding,Urochloa interespecific,Grasses,96.77,15.97,84.03,...,167.83,0.22,41.63,2.56,0.67,241.38,3588.08,3.59,21.53,12
3,F24-3585,CIAT-BR-09-1232,1.0,57.0,Breeding,Urochloa interespecific,Grasses,97.24,13.80,86.20,...,111.63,0.21,37.33,NaN,0.54,206.83,2068.33,2.07,12.41,3
4,F24-3586,CIAT-BR-12-4951,1.0,59.0,Breeding,Urochloa interespecific,Grasses,95.75,15.48,84.52,...,108.69,0.22,32.33,NaN,0.46,239.22,2392.19,2.39,14.35,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316,F26-3068,CIAT-CIAT-16320,6.0,NaN,Breeding,Urochloa interespecific,Grasses,95.48,10.75,89.25,...,222.74,0.23,NaN,NaN,1.17,190.50,1905.02,1.91,11.43,1
317,F26-3069,CIAT-CIAT-6387,6.0,NaN,Breeding,Urochloa interespecific,Grasses,95.77,11.62,88.38,...,193.20,0.23,36.00,NaN,0.63,287.44,2874.41,2.88,17.25,3
318,F26-3070,CIAT-CIAT-6780,6.0,NaN,Breeding,Urochloa interespecific,Grasses,95.23,11.29,88.71,...,119.50,0.24,NaN,NaN,0.71,165.10,1650.96,1.66,9.91,2
319,F26-3071,CIAT-GP-3207,6.0,NaN,Breeding,Urochloa interespecific,Grasses,96.28,11.79,88.21,...,174.29,0.23,NaN,NaN,0.66,258.73,2587.29,2.59,15.52,3


In [21]:
grasses_lab_yield.to_csv("/content/drive/MyDrive/lmf/output/2026_10_02_integration_of_breeding_agronomic_data_for_visualization_tool/grasses_lab_yield.csv",index=False)

# 4.0 Scatterplot

In [17]:
df = grasses_lab_yield

In [18]:
# ============================================================
# Requirements
# ============================================================
# pip install altair vl-convert-python pandas numpy
#
# vl-convert-python is required so that chart.to_json()-based
# rendering works cleanly.

import json
import itertools
import numpy as np
import pandas as pd
import altair as alt

# Disable Altair's default 5000-row safety limit, in case df is large.
alt.data_transformers.disable_max_rows()

# ============================================================
# NOTE: this assumes `df` already exists, already restricted to the
# Grasses dataset (Grasses + whatever Benchmark / Star Grass Control
# accessions are being compared against them), and includes all the
# original columns plus the new agronomy columns:
# no, requisitioner_x, tax_name, trial, field, plot, rep, X, Y,
# lab_delivery_date, sampling_date, material, requisitioner_y,
# genotype, state, comercial_name, type, biomass_g, dry_biomass_g,
# dry_matter_pct, height_cm, leaf_number, cover_m2, yield_g_m2,
# yield_kgDM_ha_cut, yield_ton_ha_cut, yield_ton_ha_year.
#
# This dataset does not include Benchmark / Star Grass Control
# accessions, so there are no extra id lists required — every row's
# color category comes straight from `tax_name` (species).
# ============================================================

# ============================================================
# Column labels (presentable names for tooltips / axis titles)
# ============================================================

column_labels = {
    # --- original columns ---
    "id": "Accession identifier",
    "id_lab": "Laboratory sample identifier",
    "tax_name": "Taxonomic name",
    "functional_group": "Functional group",
    "subset": "Experimental subset",
    "n_replicates_nutrition": "No. replicates (Nutrition)",
    "dm_percentage": "Dry matter (%)",
    "ash_dm": "Ash (% DM)",
    "om_percentage": "Organic matter (%)",
    "pc_percentage_dm": "Crude protein (% DM)",
    "adf_percentage_dm": "ADF (% DM)",
    "ndf_percentage_dm": "NDF (% DM)",
    "n_replicates_gas": "No. replicates (Gas)",
    "ch4_percentage_in_gas_8h": "Methane in Gas (8 h, %)",
    "ch4_percentage_in_gas_24h": "Methane in Gas (24 h, %)",
    "methane_intensity": "Methane intensity",
    "tddm": "TDDM (%)",
    "ch4_category": "Methane category",
    "tddm_category": "TDDM category",
    "lmf_category": "LMF category",
    "lmf_category_rank": "Position on LMF category",
    "quartile": "LMF quartile",
    "quartile_rank": "LMF ranking",
    # --- new agronomy columns ---
    "no": "Record number",
    "requisitioner_x": "Requisitioner (Nutrition)",
    "trial": "Trial",
    "field": "Field",
    "plot": "Plot",
    "rep": "Replicate",
    "X": "Plot X coordinate",
    "Y": "Plot Y coordinate",
    "lab_delivery_date": "Lab delivery date",
    "sampling_date": "Sampling date",
    "material": "Material",
    "requisitioner_y": "Requisitioner (Agronomy)",
    "genotype": "Genotype",
    "state": "State",
    "comercial_name": "Commercial name",
    "type": "Type",
    "biomass_g": "Biomass (g)",
    "dry_biomass_g": "Dry biomass (g)",
    "dry_matter_pct": "Dry matter, agronomic (%)",
    "height_cm": "Plant height (cm)",
    "leaf_number": "Leaf number",
    "cover_m2": "Ground cover (m\u00b2)",
    "yield_g_m2": "Yield (g/m\u00b2)",
    "yield_kgDM_ha_cut": "Yield per cut (kg DM/ha)",
    "yield_ton_ha_cut": "Yield per cut (t/ha)",
    "yield_ton_ha_year": "Yield per year (t/ha/year)",
}

# ============================================================
# Numeric columns available for the X / Y dropdowns
# ============================================================
# Only biologically meaningful measurement columns are offered as
# axis choices (not ids, plot coordinates, replicate numbers, etc.).

old_axis_columns = [
    "n_replicates_nutrition", "dm_percentage", "ash_dm", "om_percentage",
    "pc_percentage_dm", "adf_percentage_dm", "ndf_percentage_dm",
    "n_replicates_gas", "ch4_percentage_in_gas_8h", "ch4_percentage_in_gas_24h",
    "methane_intensity", "tddm", "lmf_category_rank", "quartile_rank",
]

new_axis_columns = [
    "biomass_g", "dry_matter_pct", "height_cm", "cover_m2",
    "yield_g_m2", "yield_kgDM_ha_cut", "yield_ton_ha_cut", "yield_ton_ha_year",
]

numeric_columns = sorted(
    col for col in (old_axis_columns + new_axis_columns) if col in df.columns
)

default_x = "tddm"
default_y = "methane_intensity"

if default_x not in numeric_columns:
    default_x = numeric_columns[0]

if default_y not in numeric_columns:
    default_y = numeric_columns[1] if len(numeric_columns) > 1 else numeric_columns[0]

# Pretty labels for each numeric column, from column_labels when
# available (falling back to a title-cased version of the column
# name) — used both for the dropdown option labels and for the
# best-effort dynamic axis title update in the browser.
pretty_labels = {
    col: column_labels.get(col, col.replace("_", " ").title())
    for col in numeric_columns
}

# Longest possible axis label (by character count) across every column
# that could ever be picked for X or Y. Vega-Lite only reserves margin
# space based on the axis title that exists *when the chart is first
# compiled/rendered* — the dropdown-driven title swap happens later, in
# JS, by editing the SVG text directly, which does NOT get Vega-Lite to
# recompute margins. So if we compiled the chart with a short title
# (e.g. the default column's label) and the user later picks a much
# longer one, the replacement text can overflow past the edge of the
# chart's reserved area and get visually clipped.
# Fix: compile the chart's initial axis titles using this longest label
# as a placeholder, so enough margin is always reserved up front no
# matter which column ends up being displayed. The JS below immediately
# swaps in the real default-column label after render, so the
# placeholder is not visible to the user in practice.
longest_pretty_label = max(pretty_labels.values(), key=len)

# ============================================================
# Tooltip columns — split into two groups so the tooltip can be
# rendered as two side-by-side columns: the fields we already had
# before, and all of the new columns added to the database.
# ============================================================

old_tooltip_columns = [
    "id", "id_lab", "tax_name", "functional_group", "subset",
    "n_replicates_nutrition", "dm_percentage", "ash_dm", "om_percentage",
    "pc_percentage_dm", "adf_percentage_dm", "ndf_percentage_dm",
    "n_replicates_gas", "ch4_percentage_in_gas_8h", "ch4_percentage_in_gas_24h",
    "methane_intensity", "tddm", "ch4_category", "tddm_category",
    "lmf_category", "lmf_category_rank", "quartile", "quartile_rank",
]

new_tooltip_columns = [
    "no", "requisitioner_x", "trial", "field", "plot", "rep", "X", "Y",
    "lab_delivery_date", "sampling_date", "material", "requisitioner_y",
    "genotype", "state", "comercial_name", "type",
    "biomass_g", "dry_biomass_g", "dry_matter_pct", "height_cm", "leaf_number",
    "cover_m2", "yield_g_m2", "yield_kgDM_ha_cut", "yield_ton_ha_cut",
    "yield_ton_ha_year",
]

old_tooltip_columns = [c for c in old_tooltip_columns if c in df.columns]
new_tooltip_columns = [c for c in new_tooltip_columns if c in df.columns]

# ============================================================
# Plot group (color category) — computed once for the whole
# dataset. Points are colored by species (tax_name). This dataset
# doesn't include Benchmark / Star Grass Control accessions, so
# there's no special-case override here — if they're added back to
# the data later, re-introduce the two .loc overrides + the two
# extra domain/color entries below.
# ============================================================

points_all = df.copy()
points_all["plot_group"] = points_all["tax_name"]

species_domain = sorted(points_all["tax_name"].dropna().unique().tolist())

# Simple, named CSS colors — matching the style/spirit of the original
# template's palette (green, cornflowerblue, orange, purple, black),
# rather than the previous auto-generated hex palette. Cycled in case
# there are more species than colors in the base list.
_species_palette = [
    "green", "cornflowerblue", "orange", "purple", "black",
    "crimson", "teal", "goldenrod", "slateblue", "brown",
    "darkcyan", "darkorange", "mediumvioletred", "olive", "steelblue",
    "chocolate", "seagreen", "indigo", "darkred", "darkgoldenrod",
]
species_colors = list(itertools.islice(itertools.cycle(_species_palette), len(species_domain)))

plot_group_order = species_domain
plot_group_colors = species_colors

# ============================================================
# Dropdown options for subset / species / category / quartile
# ============================================================

subset_options = ["All"] + sorted(points_all["subset"].astype(str).unique().tolist())
species_options = ["All"] + species_domain
category_options = ["All"] + sorted(points_all["lmf_category"].dropna().unique().tolist())
quartile_options = ["All"] + sorted(points_all["quartile"].dropna().unique().tolist())

# ============================================================
# Interactive controls
# ============================================================
# X/Y dropdowns show pretty labels but carry the actual column name
# as the underlying value, using Altair's separate options/labels
# arguments for binding_select.

x_options = [(pretty_labels[col], col) for col in numeric_columns]
y_options = [(pretty_labels[col], col) for col in numeric_columns]

sel_x = alt.param(
    name="sel_x",
    value=default_x,
    bind=alt.binding_select(options=[c for _, c in x_options], labels=[l for l, _ in x_options], name="X: ")
)

sel_y = alt.param(
    name="sel_y",
    value=default_y,
    bind=alt.binding_select(options=[c for _, c in y_options], labels=[l for l, _ in y_options], name="Y: ")
)

sel_subset = alt.param(
    name="sel_subset",
    value="All",
    bind=alt.binding_select(options=subset_options, name="Subset: ")
)

sel_species = alt.param(
    name="sel_species",
    value="All",
    bind=alt.binding_select(options=species_options, name="Species: ")
)

sel_category = alt.param(
    name="sel_category",
    value="All",
    bind=alt.binding_select(options=category_options, name="Category: ")
)

sel_quartile = alt.param(
    name="sel_quartile",
    value="All",
    bind=alt.binding_select(options=quartile_options, name="Quartile: ")
)

# ------------------------------------------------------------
# Text search box: highlight by id or id_lab, comma-separated,
# partial (substring) match, case-insensitive
# ------------------------------------------------------------

search_param = alt.param(
    name="search_id",
    value="",
    bind=alt.binding(input="text", name="Search ID or Lab ID (comma-separated): ")
)

sel_show_corr = alt.param(
    name="show_corr",
    value=False,
    bind=alt.binding_checkbox(name="Show correlation: ")
)

search_pattern_expr = (
    "replace(replace(search_id, /\\s+/g, ''), /,/g, '|')"
)

is_match_expr = (
    f"search_id !== '' && ("
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id)) || "
    f"test(regexp({search_pattern_expr}, 'i'), toString(datum.id_lab))"
    f")"
)

filter_expr = (
    "(sel_subset == 'All' || toString(datum.subset) == sel_subset) && "
    "(sel_species == 'All' || datum.tax_name == sel_species) && "
    "(sel_category == 'All' || datum.lmf_category == sel_category) && "
    "(sel_quartile == 'All' || datum.quartile == sel_quartile)"
)

# ------------------------------------------------------------
# Tooltip encoding — old fields first, then the axis values, then
# every new field. The custom JS tooltip handler below relies on
# this exact ordering (and on TOOLTIP_OLD_COUNT) to split the
# rendered tooltip into two columns.
# ------------------------------------------------------------

tooltip_old = [
    alt.Tooltip(col, title=column_labels.get(col, col.replace("_", " ").title()))
    for col in old_tooltip_columns
] + [
    alt.Tooltip("x_value:Q", title="X"),
    alt.Tooltip("y_value:Q", title="Y"),
]

tooltip_new = [
    alt.Tooltip(col, title=column_labels.get(col, col.replace("_", " ").title()))
    for col in new_tooltip_columns
]

tooltip = tooltip_old + tooltip_new
TOOLTIP_OLD_COUNT = len(tooltip_old)

# ------------------------------------------------------------
# Title row (guaranteed fallback — always shows a readable label
# for the selected columns, regardless of whether the JS-based
# axis title update below works in a given browser)
# ------------------------------------------------------------

pretty_labels_json = json.dumps(pretty_labels)

title_chart = (
    alt.Chart(pd.DataFrame({"_dummy": [1]}))
    .transform_calculate(
        label=f"({pretty_labels_json})[sel_y] + ' vs ' + ({pretty_labels_json})[sel_x]"
    )
    .mark_text(align="center", fontSize=18, fontWeight="bold", dy=0)
    .encode(text="label:N")
    .properties(width=850, height=30)
)

# ------------------------------------------------------------
# Scatter points — base layer (all non-matched points, colored by
# species / Benchmark / Star Grass Control) and highlight layer
# (only matched points, one shape per id)
# ------------------------------------------------------------

base_layer = (
    alt.Chart(points_all)
    .mark_circle(size=70)
    .transform_calculate(
        x_value="datum[sel_x]",
        y_value="datum[sel_y]",
    )
    .transform_filter(filter_expr)
    .transform_filter(f"!({is_match_expr})")
    .encode(
        x=alt.X("x_value:Q", title=longest_pretty_label),
        y=alt.Y("y_value:Q", title=longest_pretty_label),
        color=alt.Color(
            "plot_group:N",
            scale=alt.Scale(domain=plot_group_order, range=plot_group_colors),
            legend=alt.Legend(title="Species", symbolSize=70)
        ),
        tooltip=tooltip
    )
)

highlight_layer = (
    alt.Chart(points_all)
    .mark_point(filled=True, size=70, stroke="black", strokeWidth=1.5)
    .transform_calculate(
        x_value="datum[sel_x]",
        y_value="datum[sel_y]"
    )
    .transform_filter(filter_expr)
    .transform_filter(is_match_expr)
    .encode(
        x=alt.X("x_value:Q"),
        y=alt.Y("y_value:Q"),
        # Only matched rows are ever present in this layer's data, so the
        # shape scale's domain — and therefore the legend — automatically
        # lists exactly the searched accession(s), and only those. The
        # legend is hidden via JS while the search box is empty (see
        # toggleHighlightedAccessionsLegend below), same as the original
        # template.
        shape=alt.Shape(
            "id:N",
            legend=alt.Legend(title="Highlighted accession(s)", symbolSize=70, titleLimit=0)
        ),
        color=alt.value("red"),
        tooltip=tooltip
    )
)

points_chart = base_layer + highlight_layer

BIPLOT_WIDTH = 850
BIPLOT_HEIGHT = 600
biplot = points_chart.properties(width=BIPLOT_WIDTH, height=BIPLOT_HEIGHT)

# ------------------------------------------------------------
# Final layout — controls above, then title, then the plot
# ------------------------------------------------------------

final_plot = (
    alt.vconcat(title_chart, biplot)
    .add_params(sel_x, sel_y, sel_subset, sel_species, sel_category, sel_quartile, search_param, sel_show_corr)
    .configure_axis(labelFontSize=14, titleFontSize=18)
    .configure_legend(titleFontSize=16, labelFontSize=14)
    .configure_view(strokeWidth=0)
    # Extra safety-margin padding on top of whatever Vega-Lite already
    # reserves, so a swapped-in axis title (see longest_pretty_label
    # above) always has room to breathe and never gets clipped by the
    # edge of the SVG canvas.
    .properties(padding={"left": 30, "right": 10, "top": 15, "bottom": 20})
    .interactive()
)

# ============================================================
# Save as interactive HTML with controls forced above the chart
# ============================================================

# ------------------------------------------------------------
# Lightweight data used by two purely-JS features (kept separate
# from the Vega-Lite spec, small, no reliance on Vega internals):
#   1) the live correlation label (Pearson r between whichever X/Y
#      columns are selected, over rows passing the current filters)
#   2) the custom "Highlighted accession(s)" label, which recomputes
#      which ids match the search box using the exact same
#      substring/case-insensitive rule as the chart's own highlight
#      layer, and is only ever shown when that list is non-empty.
# ------------------------------------------------------------

corr_field_columns = sorted(set(["id", "id_lab", "subset", "tax_name", "lmf_category", "quartile"] + numeric_columns))
corr_field_columns = [c for c in corr_field_columns if c in points_all.columns]
corr_records = (
    points_all[corr_field_columns]
    .where(pd.notnull(points_all[corr_field_columns]), None)
    .to_dict(orient="records")
)

chart_spec = final_plot.to_json(indent=None)
pretty_labels_json = json.dumps(pretty_labels)
corr_records_json = json.dumps(corr_records)

html_template = f"""
<!DOCTYPE html>
<html>
<head>
  <meta charset="utf-8" />
  <title>Interactive Scatterplot — Grasses</title>
  <script src="https://cdn.jsdelivr.net/npm/vega@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-lite@5"></script>
  <script src="https://cdn.jsdelivr.net/npm/vega-embed@6"></script>
  <script src="https://cdn.jsdelivr.net/npm/utif@3.1.0/UTIF.js"></script>
  <script src="https://cdnjs.cloudflare.com/ajax/libs/jspdf/2.5.1/jspdf.umd.min.js"></script>
  <style>
    body {{
      font-family: sans-serif;
      display: flex;
      flex-direction: column;
      align-items: center;
    }}
    #controls {{
      margin: 20px 0;
      font-size: 16px;
    }}
    /* Visual gap between the X/Y axis controls and the rest
       (Subset, Species, Category, Quartile, Search). Controls are
       added in that order, so the 3rd one (Subset) gets pushed
       right to create the separation. Targeting .vega-bind directly
       since bindingsElement points straight at #controls (no
       intermediate .vega-bindings wrapper in that case). */
    #controls .vega-bind:nth-of-type(3) {{
      margin-left: 28px;
    }}
    #controls .vega-bindings .vega-bind:nth-of-type(3) {{
      margin-left: 28px;
    }}
    #controls > div:nth-of-type(3) {{
      margin-left: 28px;
    }}
    /* Small gap before the "Show correlation" checkbox (8th control:
       X, Y, Subset, Species, Category, Quartile, Search, Show correlation) */
    #controls .vega-bind:nth-of-type(8) {{
      margin-left: 20px;
    }}
    #controls .vega-bindings .vega-bind:nth-of-type(8) {{
      margin-left: 20px;
    }}
    #controls > div:nth-of-type(8) {{
      margin-left: 20px;
    }}
    #download-buttons {{
      margin: 20px 0 10px 0;
      display: flex;
      align-items: center;
      gap: 10px;
    }}
    #download-buttons select,
    #download-buttons button {{
      padding: 8px 16px;
      font-size: 14px;
      border: 1px solid #ccc;
      border-radius: 6px;
      background: #f7f7f9;
      cursor: pointer;
    }}
    #download-buttons button:hover {{
      background: #e9e9ee;
    }}
    #custom-tooltip {{
      position: fixed;
      pointer-events: none;
      z-index: 10000;
      background: white;
      border: 1px solid #999;
      border-radius: 6px;
      padding: 8px 10px;
      font-size: 12px;
      box-shadow: 0 2px 8px rgba(0,0,0,0.15);
      display: none;
      max-width: 90vw;
    }}
    #custom-tooltip .tt-columns {{
      display: flex;
      gap: 14px;
    }}
    #custom-tooltip .tt-col-new {{
      border-left: 1px solid #ddd;
      padding-left: 14px;
    }}
    #custom-tooltip table {{
      border-collapse: collapse;
    }}
    #custom-tooltip td {{
      white-space: nowrap;
      padding: 1px 0;
    }}
    #custom-tooltip td.tt-key {{
      padding-right: 6px;
      color: #555;
    }}
    #custom-tooltip td.tt-val {{
      font-weight: 600;
    }}
  </style>
</head>
<body>
  <div id="controls"></div>
  <div id="vis"></div>
  <div id="download-buttons">
    <select id="format-select">
      <option value="png">PNG</option>
      <option value="jpg">JPG</option>
      <option value="tiff">TIFF</option>
      <option value="pdf">PDF</option>
    </select>
    <button id="btn-download">Download</button>
  </div>
  <div id="custom-tooltip"></div>

  <script type="text/javascript">
    const spec = {chart_spec};
    const prettyLabels = {pretty_labels_json};
    const TOOLTIP_OLD_COUNT = {TOOLTIP_OLD_COUNT};
    const corrRecords = {corr_records_json};
    const BIPLOT_WIDTH = {BIPLOT_WIDTH};
    const BIPLOT_HEIGHT = {BIPLOT_HEIGHT};

    const DPI = 300;
    const DPI_SCALE = DPI / 96;

    // ----------------------------------------------------------
    // Custom two-column tooltip handler. Vega-Lite builds the
    // tooltip payload ("value") as a plain object of
    // {{ title: formattedValue }} pairs, in the same order the
    // tooltip channels were declared in — so we split that object
    // at TOOLTIP_OLD_COUNT to get the "existing fields" column and
    // the "new fields" column.
    // ----------------------------------------------------------
    function buildTooltipTable(entries) {{
      let html = "<table>";
      entries.forEach(function(pair) {{
        const k = pair[0], v = pair[1];
        html += "<tr><td class=\\"tt-key\\">" + k + "</td><td class=\\"tt-val\\">" + v + "</td></tr>";
      }});
      html += "</table>";
      return html;
    }}

    function customTooltipHandler(handler, event, item, value) {{
      const tooltipDiv = document.getElementById("custom-tooltip");

      if (value == null || (typeof value === "object" && Object.keys(value).length === 0)) {{
        tooltipDiv.style.display = "none";
        return;
      }}

      const entries = Object.entries(value);
      const oldEntries = entries.slice(0, TOOLTIP_OLD_COUNT);
      const newEntries = entries.slice(TOOLTIP_OLD_COUNT);

      tooltipDiv.innerHTML =
        '<div class="tt-columns">' +
          '<div class="tt-col-old">' + buildTooltipTable(oldEntries) + '</div>' +
          '<div class="tt-col-new">' + buildTooltipTable(newEntries) + '</div>' +
        '</div>';

      tooltipDiv.style.display = "block";

      const padding = 14;
      let x = event.clientX + padding;
      let y = event.clientY + padding;

      const rect = tooltipDiv.getBoundingClientRect();
      if (x + rect.width > window.innerWidth) {{
        x = event.clientX - rect.width - padding;
      }}
      if (y + rect.height > window.innerHeight) {{
        y = event.clientY - rect.height - padding;
      }}

      tooltipDiv.style.left = x + "px";
      tooltipDiv.style.top = y + "px";
    }}

    vegaEmbed("#vis", spec, {{
      actions: false,
      bindingsElement: "#controls",
      renderer: "svg",
      tooltip: false
    }}).then(function(result) {{

      const view = result.view;
      view.tooltip(customTooltipHandler);

      function triggerDownload(url, filename) {{
        const link = document.createElement("a");
        link.href = url;
        link.download = filename;
        document.body.appendChild(link);
        link.click();
        document.body.removeChild(link);
      }}

      function downloadAs(format) {{
        view.toCanvas(DPI_SCALE).then(function(canvas) {{

          if (format === "png") {{
            triggerDownload(canvas.toDataURL("image/png"), "scatterplot.png");

          }} else if (format === "jpg") {{
            triggerDownload(canvas.toDataURL("image/jpeg", 0.95), "scatterplot.jpg");

          }} else if (format === "tiff") {{
            const ctx = canvas.getContext("2d");
            const imageData = ctx.getImageData(0, 0, canvas.width, canvas.height);
            const tiffBuffer = UTIF.encodeImage(imageData.data, canvas.width, canvas.height);
            const blob = new Blob([tiffBuffer], {{ type: "image/tiff" }});
            const url = URL.createObjectURL(blob);
            triggerDownload(url, "scatterplot.tiff");

          }} else if (format === "pdf") {{
            const widthIn = canvas.width / DPI;
            const heightIn = canvas.height / DPI;
            const {{ jsPDF }} = window.jspdf;
            const pdf = new jsPDF({{
              orientation: widthIn >= heightIn ? "landscape" : "portrait",
              unit: "in",
              format: [widthIn, heightIn]
            }});
            pdf.addImage(canvas.toDataURL("image/png"), "PNG", 0, 0, widthIn, heightIn);
            pdf.save("scatterplot.pdf");
          }}

        }}).catch(console.error);
      }}

      document.getElementById("btn-download").addEventListener("click", function() {{
        const format = document.getElementById("format-select").value;
        downloadAs(format);
      }});

      // ----------------------------------------------------------
      // Shared helper: reliably tell apart the x-axis group from the
      // y-axis group among whatever matches
      // '[aria-roledescription="axis"], g.role-axis'. Rather than
      // trusting rotation/transform heuristics (which turned out to
      // mislabel — and thus swap — the axis titles), this just picks
      // the WIDEST group as the x-axis (it runs along the plot's
      // width) and the TALLEST as the y-axis (it runs along the
      // plot's height). That's true regardless of chart size/zoom.
      // ----------------------------------------------------------
      // Reliably tell apart the x-axis group from the y-axis group.
      // Verified directly against Vega's real SVG output: axis groups
      // carry aria-roledescription="axis" AND an aria-label that
      // literally starts with "X-axis" or "Y-axis" — e.g.
      // aria-label="X-axis titled 'TDDM (%)' for a linear scale...".
      // (An earlier version of this also matched via a `g.role-axis`
      // class selector, which turned out to ALSO catch Vega's hidden,
      // text-less gridline groups — those don't carry
      // aria-roledescription at all, and mixing them into a
      // width/height comparison is what caused the axis titles to
      // get swapped.)
      // ----------------------------------------------------------
      function findAxisRoots() {{
        const container = view.container();
        if (!container) return null;

        const axisGroups = Array.from(
          container.querySelectorAll('[aria-roledescription="axis"]')
        );

        let xAxisRoot = null, yAxisRoot = null;
        axisGroups.forEach(function(g) {{
          const label = g.getAttribute("aria-label") || "";
          if (label.indexOf("X-axis") === 0) xAxisRoot = g;
          else if (label.indexOf("Y-axis") === 0) yAxisRoot = g;
        }});

        if (!xAxisRoot || !yAxisRoot) return null;
        return {{ xAxisRoot: xAxisRoot, yAxisRoot: yAxisRoot }};
      }}

      // BEST EFFORT: update the actual x/y axis titles to show the
      // pretty column label (from column_labels) for whatever is
      // currently selected. The raw "Y vs X" text is ALSO shown in
      // the chart's main title above, which does not depend on this
      // and always works.
      function updateAxisTitles() {{
        const xCol = view.signal("sel_x");
        const yCol = view.signal("sel_y");
        const xLabel = prettyLabels[xCol] || xCol;
        const yLabel = prettyLabels[yCol] || yCol;

        const roots = findAxisRoots();
        if (!roots) return;

        // Verified against Vega's real SVG output: each axis root
        // contains exactly one <g class="role-axis-title"> wrapping
        // exactly one <text> — this is the title, directly, no
        // guessing needed.
        function setTitleFor(root, label) {{
          const titleText = root.querySelector("g.role-axis-title text");
          if (titleText) titleText.textContent = label;
        }}

        setTitleFor(roots.xAxisRoot, xLabel);
        setTitleFor(roots.yAxisRoot, yLabel);
      }}

      setTimeout(updateAxisTitles, 100);
      setTimeout(updateAxisTitles, 300);
      view.addSignalListener("sel_x", function() {{
        setTimeout(updateAxisTitles, 100);
        setTimeout(updateAxisTitles, 300);
      }});
      view.addSignalListener("sel_y", function() {{
        setTimeout(updateAxisTitles, 100);
        setTimeout(updateAxisTitles, 300);
      }});

      // ----------------------------------------------------------
      // Explicitly hide the "Highlighted accession(s)" legend while
      // the search box is empty — same approach as the original
      // template, using the verified-correct legend selector
      // (aria-roledescription="legend").
      // ----------------------------------------------------------
      function toggleHighlightedAccessionsLegend(searchValue) {{
        const container = view.container();
        if (!container) return;

        const isActive = !!(searchValue && searchValue.trim() !== "");

        const legendGroups = container.querySelectorAll('[aria-roledescription="legend"]');
        legendGroups.forEach(function(g) {{
          const texts = g.querySelectorAll("text");
          let isTargetLegend = false;
          texts.forEach(function(t) {{
            if (t.textContent.trim().indexOf("Highlighted") === 0) {{
              isTargetLegend = true;
            }}
          }});
          if (isTargetLegend) {{
            g.style.display = isActive ? "" : "none";
          }}
        }});
      }}

      [0, 100, 300].forEach(function(delay) {{
        setTimeout(function() {{
          toggleHighlightedAccessionsLegend(view.signal("search_id"));
        }}, delay);
      }});
      view.addSignalListener("search_id", function(name, value) {{
        setTimeout(function() {{ toggleHighlightedAccessionsLegend(value); }}, 50);
        setTimeout(function() {{ toggleHighlightedAccessionsLegend(value); }}, 250);
      }});

      // Belt-and-suspenders: re-apply the hide/show rule the instant
      // Vega adds or replaces any DOM node under the chart (e.g. when
      // it redraws the legend after a filter change), instead of only
      // on a fixed delay. This closes any remaining gap where the
      // empty "Highlighted accession(s)" title could flash visible
      // for a moment before/after a re-render.
      const highlightLegendObserver = new MutationObserver(function() {{
        toggleHighlightedAccessionsLegend(view.signal("search_id"));
      }});
      const chartContainer = view.container();
      if (chartContainer) {{
        highlightLegendObserver.observe(chartContainer, {{ childList: true, subtree: true }});
      }}

      // ----------------------------------------------------------
      // Live correlation label: Pearson r between whatever X/Y
      // columns are selected, computed over whatever rows currently
      // pass the Subset/Species/Category/Quartile filters. Shown as
      // a small floating label anchored just below the "Species"
      // legend — i.e. right before the "Highlighted accession(s)"
      // legend underneath it.
      // ----------------------------------------------------------

      function collectFilteredXY() {{
        const subset = String(view.signal("sel_subset"));
        const species = String(view.signal("sel_species"));
        const category = String(view.signal("sel_category"));
        const quartile = String(view.signal("sel_quartile"));
        const xCol = view.signal("sel_x");
        const yCol = view.signal("sel_y");

        const xs = [];
        const ys = [];

        corrRecords.forEach(function(d) {{
          if (subset !== "All" && String(d.subset) !== subset) return;
          if (species !== "All" && d.tax_name !== species) return;
          if (category !== "All" && d.lmf_category !== category) return;
          if (quartile !== "All" && String(d.quartile) !== quartile) return;

          const xv = d[xCol];
          const yv = d[yCol];
          if (xv === null || yv === null || xv === undefined || yv === undefined) return;
          if (isNaN(xv) || isNaN(yv)) return;

          xs.push(+xv);
          ys.push(+yv);
        }});

        return {{ xs: xs, ys: ys }};
      }}

      function pearsonCorrelation(xs, ys) {{
        const n = xs.length;
        if (n < 2) return null;

        const meanX = xs.reduce(function(a, b) {{ return a + b; }}, 0) / n;
        const meanY = ys.reduce(function(a, b) {{ return a + b; }}, 0) / n;

        let num = 0, denX = 0, denY = 0;
        for (let i = 0; i < n; i++) {{
          const dx = xs[i] - meanX;
          const dy = ys[i] - meanY;
          num += dx * dy;
          denX += dx * dx;
          denY += dy * dy;
        }}

        if (denX === 0 || denY === 0) return null;
        return num / Math.sqrt(denX * denY);
      }}

      // Finds the plotting area's real pixel bounds directly, using
      // its own background rectangle as ground truth — verified
      // against Vega's real SVG output: the biplot's cell is drawn as
      // a <path class="background" d="M0,0h850v600h-850Z">, i.e. its
      // "d" attribute encodes exactly BIPLOT_WIDTH/BIPLOT_HEIGHT. This
      // sidesteps axis-group heuristics entirely for this purpose.
      function getPlotPixelBounds() {{
        const container = view.container();
        if (!container) return null;

        const svg = container.querySelector("svg");
        if (!svg) return null;

        const marker = "h" + BIPLOT_WIDTH + "v" + BIPLOT_HEIGHT;
        const backgrounds = Array.from(container.querySelectorAll("path.background"));
        const plotBg = backgrounds.find(function(p) {{
          const d = p.getAttribute("d") || "";
          return d.indexOf(marker) !== -1;
        }});
        if (!plotBg) return null;

        const rect = plotBg.getBoundingClientRect();
        return {{
          svg: svg,
          plotLeft: rect.left,
          plotTop: rect.top,
          plotRight: rect.right,
          plotBottom: rect.bottom
        }};
      }}

      function clientPointToSvgPoint(svg, clientX, clientY) {{
        const ctm = svg.getScreenCTM();
        if (!ctm) return null;
        const pt = svg.createSVGPoint();
        pt.x = clientX;
        pt.y = clientY;
        return pt.matrixTransform(ctm.inverse());
      }}

      function updateCorrelationLabel() {{
        const container = view.container();
        if (!container) return;

        // Always drop the previous label first — we recreate it fresh
        // on every call, which sidesteps any stale positioning/visibility
        // left over from before the chart re-rendered.
        const existing = container.querySelector("#corr-label-svg");
        if (existing) existing.remove();

        const showCorr = view.signal("show_corr");
        if (!showCorr) return;

        const bounds = getPlotPixelBounds();
        if (!bounds) return;

        const {{ xs, ys }} = collectFilteredXY();
        const r = pearsonCorrelation(xs, ys);
        const labelText = (r === null)
          ? "Correlation (r): n/a"
          : ("Correlation (r): " + r.toFixed(2) + " (n=" + xs.length + ")");

        // Fixed pixel padding inward from the plot's left/bottom edges,
        // so the label always sits inside the plotting area — clear of
        // both axes and the chart's outer edge, however big the chart is.
        const padLeft = 46;
        const padBottom = 34;

        const svgPoint = clientPointToSvgPoint(
          bounds.svg,
          bounds.plotLeft + padLeft,
          bounds.plotBottom - padBottom
        );
        if (!svgPoint) return;

        const svgNS = "http://www.w3.org/2000/svg";
        const textEl = document.createElementNS(svgNS, "text");
        textEl.setAttribute("id", "corr-label-svg");
        textEl.setAttribute("x", svgPoint.x);
        textEl.setAttribute("y", svgPoint.y);
        textEl.setAttribute("font-size", "13");
        textEl.setAttribute("font-weight", "600");
        textEl.setAttribute("fill", "#333");
        textEl.textContent = labelText;

        bounds.svg.appendChild(textEl);
      }}

      setTimeout(updateCorrelationLabel, 150);
      ["sel_x", "sel_y", "sel_subset", "sel_species", "sel_category", "sel_quartile", "show_corr"].forEach(function(sig) {{
        view.addSignalListener(sig, function() {{
          setTimeout(updateCorrelationLabel, 100);
          setTimeout(updateCorrelationLabel, 300);
        }});
      }});
      window.addEventListener("resize", function() {{
        setTimeout(updateCorrelationLabel, 50);
      }});

    }}).catch(console.error);
  </script>
</body>
</html>
"""

with open("scatterplot_grasses.html", "w", encoding="utf-8") as f:
    f.write(html_template)

print("Done: scatterplot_grasses.html")

Done: scatterplot_grasses.html


In [ ]:
with open("scatterplot_grasses.html", "w", encoding="utf-8") as f:
    f.write(html_template)

In [20]:
with open("/content/drive/MyDrive/lmf/output/2026_10_02_integration_of_breeding_agronomic_data_for_visualization_tool/scatterplot_grasses.html", "w", encoding="utf-8") as f:
    f.write(html_template)